In [ ]:
%pip install google-generativeai

In [ ]:
%pip install pandasai

In [ ]:
%pip install pandasai[langchain]

In [ ]:
%pip install pandas

In [ ]:
%pip install langchain_google_genai

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import os

In [ ]:
if os.environ.get("GOOGLE_API_KEY"):
    print("Successfully loaded")

In [ ]:
%pip install openpyxl

In [ ]:
import pandas as pd
from pandasai import SmartDataframe
from langchain_google_genai import GoogleGenerativeAI


# Sample DataFrame
df = pd.read_excel("orders.xlsx", sheet_name="Sheet1", engine="openpyxl")

# Instantiate a LLM
llm = GoogleGenerativeAI(
    google_api_key=os.environ.get("GOOGLE_API_KEY"), model="gemini-pro", temperature=0.1
)

In [ ]:
llm.invoke("Who is Elon Musk?")

In [ ]:
df.head()

In [ ]:
df_chain = SmartDataframe(df, config={"llm": llm})

In [ ]:
df_chain.chat('How many orders are successfully delivered?')

In [ ]:
df_chain.chat("Refer the customer_id column and find which customer has placed the highest number of orders?")

In [ ]:
df.groupby("customer_id").agg({
    "order_id": "count"
}).sort_values("order_id", ascending=False)

In [ ]:
df_chain.chat("Visualize the month wise placement of orders")

In [ ]:
df_chain.chat("""Visualize the month wise placement of orders. 
              Please sort the months based on the month numbers such as first chart should be of first 
              month that is january, then february, and so on..""")

In [ ]:
df["month"] = df["order_purchase_date"].dt.month_name()

In [ ]:
df.groupby("month").agg({
    "order_id": "count"
})

In [ ]:
df_chain.chat("""
              Visualize that over the period placement of order is increasing or decreasing.
              Consider the month columns and based on that plot a graph that shows the orders placement
              over the month. Ex: in jane if 25 Orders are placed, in february 20 orders are placed, so orders
              has decaresed, so plot a graph that shows this clearly.
              
              Plot a line chart starting with the January month and showing the placement of orders. 
              Consider the label as vertical on x-axis.

              Visualise the data month number wise, plecment of order of 1 month, then of 2 month, and so on..
              use the dt.month method and then sort it in ascending order, and based on that visualize the data
              """)

#### analysing multiple sheets

In [ ]:
import pandas as pd

customers = pd.read_excel("globalmart-business-data.xlsx", sheet_name="customers", engine="openpyxl")
orders = pd.read_excel("globalmart-business-data.xlsx", sheet_name="orders", engine="openpyxl")
transactions = pd.read_excel("globalmart-business-data.xlsx", sheet_name="transactions", engine="openpyxl")
products = pd.read_excel("globalmart-business-data.xlsx", sheet_name="products", engine="openpyxl")

In [ ]:
from pandasai import SmartDatalake

# excel_file = "./globalmart-business-data.xlsx"

excel_analysis_chain = SmartDatalake([customers, orders, transactions, products], config={"llm": llm})

In [ ]:
excel_analysis_chain.chat("Which customers hasn't placed any order?")

In [ ]:
excel_analysis_chain.chat("""
                          Visualize the month wise sales data. The sales data is present in transactions
                          and orders contains the data on which orders were placed on. So combine both of them
                          and analyse the month wise sales like what are the total sales in 1st Month, then in 2 Month,
                          and so on..
                          
                          Once done, visualize the analysed data""")

In [ ]:
excel_analysis_chain.chat("""
                          Visualize the month wise sales data. The sales data is present in transactions
                          and orders contains the data on which orders were placed on. So combine both of them
                          and analyse the month wise sales like what are the total sales in 1st Month, then in 2 Month,
                          and so on..
                          
                          Once done, visualize the analysed data.""")

In [ ]:
sales_df = orders.merge(transactions, on="order_id", how="inner")

In [ ]:
sales_df.columns

In [ ]:
sales_df["month"] = sales_df["order_purchase_date"].dt.month

In [ ]:
gorupby_month_sales = sales_df.groupby("month")["sales"].sum().reset_index()

In [ ]:
import matplotlib.pyplot as plt

plt.plot(gorupby_month_sales["month"], gorupby_month_sales["sales"], label="Month-wise sales")